# Q5: Lipid Metabolism, Myelin Lipid Synthesis, and Glymphatic Surrogates

This notebook asks whether Cx47 (GJC2) expression in oligodendrocytes correlates with lipid metabolic programs across cell types (oligodendrocytes, astrocytes, microglia, endothelial cells), and whether glymphatic surrogate markers (AQP4, ion homeostasis, BBB integrity) change in step with pannexin/connexin remodeling across MS lesion classes.

**Dataset:** Jäkel et al. (human MS snRNA-seq, `jakel_et_al.h5ad`)  
**Progression axis:** Lesion class (Ctrl → NAWM → A → CA → CI → RM)

**Key analyses:**
1. Lipid metabolism and myelin lipid synthesis in oligodendrocytes — correlation with GJC2
2. Cholesterol/lipid clearance in microglia (TREM2, APOE, ABCA1, CD36) across lesion stages
3. Astrocytic glymphatic surrogates: AQP4, ion homeostasis (KCNJ10, ATP1A2), BBB-related genes
4. Lymphatic endothelial markers in endothelial/vascular cells
5. Cross-compartment lesion-level correlation with GJC2

> **Honest framing:** scRNA-seq cannot measure AQP4 polarization to endfeet (the functional glymphatic marker) or lymph flow. These analyses provide gene-expression-level evidence that can motivate orthogonal imaging validation.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr
from IPython.display import Markdown, display

CANDIDATES = [Path.cwd().resolve(), Path.cwd().resolve().parent]
REPO_ROOT = next(
    (p for p in CANDIDATES if (p / "src").exists() and (p / "config").exists()), None
)
if REPO_ROOT is None:
    raise RuntimeError("Could not locate the repository root.")

MPL_CACHE_DIR = REPO_ROOT / ".cache" / "matplotlib"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from cx47_oligo.gene_panels import GENE_PANELS
from cx47_oligo.exports import ensure_results_dir, save_current_figure, save_json, save_table
from cx47_oligo.h5ad_tools import (
    adata_overview,
    expression_frame,
    load_h5ad,
    obs_column_summary,
    panel_availability_table,
    resolve_gene_symbols,
    score_gene_panel,
)
from cx47_oligo.scanpy_tools import obs_keyword_mask

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#fbfcfd",
    "axes.edgecolor": "#d7dde2",
    "grid.color": "#dde4ea",
    "grid.alpha": 0.75,
})

In [ ]:
DATASET_PATH = REPO_ROOT / "data" / "raw" / "jakel_et_al.h5ad"
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Missing: {DATASET_PATH}")

RESULTS_DIR = ensure_results_dir(REPO_ROOT, "05_lipid_glymphatic", DATASET_PATH)
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

CELLTYPE_COLUMN = "Celltypes"
LESION_COLUMN = "Lesion"
SAMPLE_COLUMN = "Sample"
LESION_ORDER = ["Ctrl", "NAWM", "A", "CA", "CI", "RM"]

# Panels by compartment
OLIGO_PANELS  = ["lipid_metabolism", "myelin_lipid_synthesis", "oligodendrocyte_identity", "panglial_connexins"]
MICRO_PANELS  = ["lipid_metabolism", "microglial_activation"]
ASTRO_PANELS  = ["lipid_metabolism", "ion_homeostasis", "astrocyte_identity", "bbb_related"]
ENDO_PANELS   = ["lymphatic_endothelial", "lymphatic_barrier", "bbb_related"]
ALL_PANELS    = list(dict.fromkeys(OLIGO_PANELS + MICRO_PANELS + ASTRO_PANELS + ENDO_PANELS))

# Individual lipid-relevant genes to highlight per compartment
MICRO_HIGHLIGHT_GENES = ["TREM2", "APOE", "ABCA1", "CD36", "LPL", "PLIN2"]
ASTRO_HIGHLIGHT_GENES = ["AQP4", "KCNJ10", "ATP1A2", "APOE", "SLC1A2"]
OLIGO_HIGHLIGHT_GENES = ["GJC2", "GJB1", "CERS2", "UGT8", "HMGCR", "FASN", "ABCA1"]

print("Dataset:", DATASET_PATH)
print("Results:", RESULTS_DIR)

In [ ]:
adata = load_h5ad(DATASET_PATH)
display(adata_overview(adata))

present_lesions = set(adata.obs[LESION_COLUMN].astype(str).unique())
LESION_ORDER_PRESENT = [l for l in LESION_ORDER if l in present_lesions]
adata.obs["Lesion_ordered"] = pd.Categorical(
    adata.obs[LESION_COLUMN].astype(str),
    categories=LESION_ORDER_PRESENT,
    ordered=True,
)

meta_cols = [c for c in [CELLTYPE_COLUMN, LESION_COLUMN, SAMPLE_COLUMN, "Condition"] if c in adata.obs.columns]
display(obs_column_summary(adata).loc[lambda df: df["column"].isin(meta_cols)])

coverage = panel_availability_table(adata, ALL_PANELS)
display(coverage)
save_table(coverage, RESULTS_DIR / "panel_coverage.csv", index=False)

# Cell-type subsets
oligo_mask  = obs_keyword_mask(adata, CELLTYPE_COLUMN, ["oligo", "opc", "cop", "imolg"])
astro_mask  = obs_keyword_mask(adata, CELLTYPE_COLUMN, ["astro"])
micro_mask  = obs_keyword_mask(adata, CELLTYPE_COLUMN, ["microglia", "macrophage"])
endo_mask   = obs_keyword_mask(adata, CELLTYPE_COLUMN, ["endothelial", "vasc", "pericyte"])

adata_oligo = adata[oligo_mask].copy()
adata_astro = adata[astro_mask].copy()
adata_micro = adata[micro_mask].copy()
adata_endo  = adata[endo_mask].copy()

print(f"Oligos: {adata_oligo.n_obs}  Astrocytes: {adata_astro.n_obs}  Microglia: {adata_micro.n_obs}  Endothelial: {adata_endo.n_obs}")
save_json(
    {
        "dataset_path": str(DATASET_PATH),
        "results_dir": str(RESULTS_DIR),
        "lesion_order": LESION_ORDER_PRESENT,
        "oligo_panels": OLIGO_PANELS,
        "micro_panels": MICRO_PANELS,
        "astro_panels": ASTRO_PANELS,
        "endo_panels": ENDO_PANELS,
    },
    RESULTS_DIR / "run_metadata.json",
)

## Helpers

In [ ]:
def score_panels(adata_sub, panels):
    scored = {}
    for panel in panels:
        try:
            score, _ = score_gene_panel(adata_sub, panel, normalize_counts=False)
            adata_sub.obs[score.name] = score
            scored[panel] = score.name
        except ValueError as exc:
            display(Markdown(f"**{panel}** skipped: {exc}"))
    return scored


def mean_by_lesion(adata_sub, cols, lesion_col="Lesion_ordered", order=None):
    cols = [c for c in cols if c in adata_sub.obs.columns]
    frame = adata_sub.obs.groupby(lesion_col, observed=True)[cols].mean()
    if order:
        frame = frame.reindex([l for l in order if l in frame.index])
    return frame


def zscore_frame(frame):
    return frame.apply(lambda s: (s - s.mean()) / (s.std(ddof=0) if s.std(ddof=0) else 1.0))


def plot_heatmap(frame, title, cmap="mako"):
    fig, ax = plt.subplots(figsize=(max(6, 1.1 * len(frame.columns) + 2), max(4, 0.55 * len(frame.index) + 2)))
    sns.heatmap(frame, cmap=cmap, linewidths=0.4, linecolor="white", ax=ax)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("")
    plt.xticks(rotation=35, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()


def spearman_table(frame, x_cols, y_cols):
    rows = []
    for x in x_cols:
        for y in y_cols:
            if x == y:
                continue
            valid = frame[[x, y]].dropna()
            if len(valid) < 3 or valid[x].nunique() < 2 or valid[y].nunique() < 2:
                rows.append({"x": x, "y": y, "rho": np.nan, "pvalue": np.nan, "n": len(valid)})
                continue
            rho, pvalue = spearmanr(valid[x], valid[y])
            rows.append({"x": x, "y": y, "rho": round(rho, 3), "pvalue": round(pvalue, 4), "n": len(valid)})
    return pd.DataFrame(rows)


def add_gene_expr(adata_sub, genes):
    """Add individual gene expression columns to adata_sub.obs. Returns resolved gene names."""
    expr = expression_frame(adata_sub, genes, normalize_counts=False)
    for gene in expr.columns:
        adata_sub.obs[f"{gene}_expr"] = expr[gene].values
    return list(expr.columns)

## Oligodendrocyte Lipid Metabolism and Myelin Lipid Synthesis by Lesion Class

Myelin is predominantly lipid. Remyelination requires de novo synthesis of ceramides (CERS2), sulfatides (UGT8), and fatty acids (FASN). We ask whether these programs track GJC2 recovery across lesion stages.

In [ ]:
oligo_scored = score_panels(adata_oligo, OLIGO_PANELS)
oligo_genes_resolved = add_gene_expr(adata_oligo, OLIGO_HIGHLIGHT_GENES)

oligo_feature_cols = list(oligo_scored.values()) + [f"{g}_expr" for g in oligo_genes_resolved]
oligo_by_lesion = mean_by_lesion(adata_oligo, oligo_feature_cols, order=LESION_ORDER_PRESENT)
display(oligo_by_lesion.round(3))
save_table(oligo_by_lesion, RESULTS_DIR / "oligo_lipid_scores_by_lesion.csv")

plot_heatmap(zscore_frame(oligo_by_lesion), "Oligodendrocyte lipid and connexin programs by lesion (z-scored)", cmap="coolwarm")
#save_current_figure(FIGURES_DIR / "oligo_lipid_scores_by_lesion_zscored.png")

# Percent expressing GJC2 and CERS2 per lesion class
for gene in ["GJC2", "CERS2", "UGT8"]:
    col = f"{gene}_expr"
    if col in adata_oligo.obs.columns:
        pct = (
            adata_oligo.obs.assign(pos=(adata_oligo.obs[col] > 0))
            .groupby("Lesion_ordered", observed=True)["pos"]
            .mean()
            .mul(100)
            .reindex(LESION_ORDER_PRESENT)
            .rename(f"{gene}_pct_positive")
        )
        display(pct.to_frame().round(1))
        save_table(pct.to_frame(), RESULTS_DIR / f"oligo_{gene.lower()}_pct_expressing_by_lesion.csv")

## Microglial Lipid Clearance (TREM2/APOE/CD36 Axis) by Lesion Class

Myelin debris phagocytosis by microglia is rate-limiting for remyelination. TREM2 and APOE are key regulators of lipid-loaded microglial states. We check whether the lipid clearance signature in microglia peaks at active/CA lesions and whether it correlates with the GJC2 recovery in oligodendrocytes at the lesion-class level.

In [ ]:
if adata_micro.n_obs == 0:
    display(Markdown("**No microglial cells found.** Adjust keyword filter if needed."))
else:
    micro_scored = score_panels(adata_micro, MICRO_PANELS)
    micro_genes_resolved = add_gene_expr(adata_micro, MICRO_HIGHLIGHT_GENES)

    micro_feature_cols = list(micro_scored.values()) + [f"{g}_expr" for g in micro_genes_resolved]
    micro_by_lesion = mean_by_lesion(adata_micro, micro_feature_cols, order=LESION_ORDER_PRESENT)
    display(micro_by_lesion.round(3))
    save_table(micro_by_lesion, RESULTS_DIR / "microglia_lipid_scores_by_lesion.csv")

    plot_heatmap(zscore_frame(micro_by_lesion), "Microglial lipid clearance programs by lesion (z-scored)", cmap="Oranges")
    #save_current_figure(FIGURES_DIR / "microglia_lipid_scores_by_lesion_zscored.png")

    # Highlight TREM2 trajectory
    if "TREM2_expr" in adata_micro.obs.columns:
        trem2_by_lesion = (
            adata_micro.obs.groupby("Lesion_ordered", observed=True)["TREM2_expr"]
            .agg(["mean", lambda s: (s > 0).mean() * 100])
            .rename(columns={"mean": "mean_expression", "<lambda_0>": "pct_positive"})
            .reindex(LESION_ORDER_PRESENT)
        )
        display(Markdown("**TREM2 in microglia by lesion:**"))
        display(trem2_by_lesion.round(2))
        save_table(trem2_by_lesion, RESULTS_DIR / "microglia_trem2_by_lesion.csv")

## Astrocytic Glymphatic Surrogates and BBB Markers by Lesion Class

We use AQP4, KCNJ10, and ATP1A2 as transcriptional surrogates for astrocyte ion/water homeostasis relevant to glymphatic function. Note that AQP4 polarization to astrocyte endfeet cannot be measured from transcript data — this is an expression-level surrogate only.

In [ ]:
if adata_astro.n_obs == 0:
    display(Markdown("**No astrocyte cells found.** Adjust keyword filter if needed."))
else:
    astro_scored = score_panels(adata_astro, ASTRO_PANELS)
    astro_genes_resolved = add_gene_expr(adata_astro, ASTRO_HIGHLIGHT_GENES)

    astro_feature_cols = list(astro_scored.values()) + [f"{g}_expr" for g in astro_genes_resolved]
    astro_by_lesion = mean_by_lesion(adata_astro, astro_feature_cols, order=LESION_ORDER_PRESENT)
    display(astro_by_lesion.round(3))
    save_table(astro_by_lesion, RESULTS_DIR / "astrocyte_glymphatic_scores_by_lesion.csv")

    plot_heatmap(zscore_frame(astro_by_lesion), "Astrocyte lipid / ion homeostasis / BBB programs by lesion (z-scored)", cmap="Greens")
    #save_current_figure(FIGURES_DIR / "astrocyte_glymphatic_scores_by_lesion_zscored.png")

    # AQP4 specifically: mean expression and % expressing
    if "AQP4_expr" in adata_astro.obs.columns:
        aqp4_by_lesion = (
            adata_astro.obs.groupby("Lesion_ordered", observed=True)["AQP4_expr"]
            .agg(["mean", lambda s: (s > 0).mean() * 100])
            .rename(columns={"mean": "mean_expression", "<lambda_0>": "pct_positive"})
            .reindex(LESION_ORDER_PRESENT)
        )
        display(Markdown("**AQP4 in astrocytes by lesion:**"))
        display(aqp4_by_lesion.round(2))
        save_table(aqp4_by_lesion, RESULTS_DIR / "astrocyte_aqp4_by_lesion.csv")

## Endothelial / Vascular Lymphatic Marker Expression by Lesion Class

Lymphatic endothelial markers (PROX1, LYVE1, PDPN, VEGFC) and BBB junction genes (CLDN5, OCLN, TJP1) are scored in the vascular compartment. These are the closest expression-level proxy for glymphatic outflow and meningeal lymphatic drainage available in this dataset.

In [ ]:
if adata_endo.n_obs == 0:
    display(Markdown("**No endothelial/vascular cells found.** Adjust keyword filter if needed."))
else:
    endo_scored = score_panels(adata_endo, ENDO_PANELS)

    endo_feature_cols = list(endo_scored.values())
    endo_by_lesion = mean_by_lesion(adata_endo, endo_feature_cols, order=LESION_ORDER_PRESENT)
    display(endo_by_lesion.round(3))
    save_table(endo_by_lesion, RESULTS_DIR / "endothelial_lymphatic_scores_by_lesion.csv")

    plot_heatmap(zscore_frame(endo_by_lesion), "Endothelial lymphatic and BBB programs by lesion (z-scored)", cmap="Blues")
    #save_current_figure(FIGURES_DIR / "endothelial_lymphatic_scores_by_lesion_zscored.png")

    # Resolve individual lymphatic endothelial marker genes
    endo_lym_genes = ["PROX1", "LYVE1", "PDPN", "VEGFC", "FLT4"]
    endo_lym_resolved = add_gene_expr(adata_endo, endo_lym_genes)
    if endo_lym_resolved:
        endo_gene_by_lesion = mean_by_lesion(
            adata_endo, [f"{g}_expr" for g in endo_lym_resolved], order=LESION_ORDER_PRESENT
        )
        display(endo_gene_by_lesion.round(3))
        save_table(endo_gene_by_lesion, RESULTS_DIR / "endothelial_lymphatic_genes_by_lesion.csv")

## Cross-Compartment Correlation: GJC2 vs Lipid / Glymphatic Programs Across Lesion Classes

We build a lesion-level summary table merging all compartments and compute Spearman correlations between GJC2 expression in oligodendrocytes and each lipid/glymphatic program in other compartments. The hypothesis is that GJC2 recovery toward RM co-occurs with resolved lipid clearance in microglia and restored ion homeostasis in astrocytes.

In [ ]:
frames_to_merge = []
for label, frame in [
    ("oligo", oligo_by_lesion if "oligo_by_lesion" in dir() and not oligo_by_lesion.empty else None),
    ("micro", micro_by_lesion if "micro_by_lesion" in dir() and not micro_by_lesion.empty else None),
    ("astro", astro_by_lesion if "astro_by_lesion" in dir() and not astro_by_lesion.empty else None),
    ("endo",  endo_by_lesion  if "endo_by_lesion"  in dir() and not endo_by_lesion.empty  else None),
]:
    if frame is not None:
        frames_to_merge.append(frame.add_prefix(f"{label}__"))

if frames_to_merge:
    cross_lesion = pd.concat(frames_to_merge, axis=1).dropna(how="all")
    cross_lesion.index.name = "Lesion"
    display(cross_lesion.round(3))
    save_table(cross_lesion, RESULTS_DIR / "cross_compartment_lesion_summary.csv")
else:
    print("No compartment data available.")
    cross_lesion = pd.DataFrame()

In [ ]:
if not cross_lesion.empty:
    gjc2_col = [c for c in cross_lesion.columns if "GJC2_expr" in c]
    lipid_lipid_cols = [
        c for c in cross_lesion.columns
        if any(k in c for k in ["lipid", "myelin", "aqp4", "KCNJ10", "ATP1A2", "ion_homeostasis", "trem2", "TREM2", "lymphatic", "bbb"])
        and c not in gjc2_col
    ]

    if gjc2_col and lipid_lipid_cols:
        corr_table = spearman_table(cross_lesion, gjc2_col, lipid_lipid_cols)
        display(corr_table.sort_values("rho", ascending=False))
        save_table(corr_table, RESULTS_DIR / "gjc2_vs_lipid_glymphatic_spearman.csv", index=False)

    # Full heatmap of all cross-compartment correlations
    corr_matrix = cross_lesion.corr(method="spearman")
    plot_heatmap(corr_matrix, "Full cross-compartment Spearman correlation (lesion-averaged)", cmap="RdBu_r")
    #save_current_figure(FIGURES_DIR / "cross_compartment_spearman_heatmap.png")

## Donor-Level Variation: GJC2 and Lipid Program Co-Recovery

Sample-level aggregation to separate donor heterogeneity from lesion-class effects.

In [ ]:
oligo_sample_cols = [
    c for c in ["GJC2_expr", "lipid_metabolism_score", "myelin_lipid_synthesis_score"]
    if c in adata_oligo.obs.columns
]

if oligo_sample_cols:
    oligo_sample = (
        adata_oligo.obs
        .groupby(["Lesion_ordered", SAMPLE_COLUMN], observed=True)[oligo_sample_cols]
        .mean()
        .reset_index()
    )
    save_table(oligo_sample, RESULTS_DIR / "oligo_lipid_gjc2_sample_means.csv", index=False)

    n_cols = len(oligo_sample_cols)
    fig, axes = plt.subplots(1, n_cols, figsize=(6 * n_cols, 5))
    if n_cols == 1:
        axes = [axes]
    for ax, col in zip(axes, oligo_sample_cols):
        sns.boxplot(data=oligo_sample, x="Lesion_ordered", y=col, ax=ax, showfliers=False, color="#4a7fa5")
        sns.stripplot(data=oligo_sample, x="Lesion_ordered", y=col, ax=ax, color="#d88c32", alpha=0.8, size=6)
        ax.set_title(col.replace("_", " "))
        ax.set_xlabel("Lesion class")
        ax.set_ylabel("Mean per sample")
        ax.tick_params(axis="x", rotation=30)
    plt.tight_layout()
    #save_current_figure(FIGURES_DIR / "oligo_lipid_gjc2_by_sample_lesion.png")